# Programação Paralela — Laboratório Completo de OpenMP

Este notebook é o caderno prático da parte de **OpenMP** da disciplina de Programação Paralela.

Em cada tópico, o aluno deve:

1. entender o problema;
2. executar um exemplo mínimo;
3. testar uma aplicação;
4. validar a corretude;
5. medir desempenho;
6. modificar parâmetros;
7. explicar o comportamento observado;
8. resolver exercícios progressivos.

> **Regra do laboratório:** primeiro corretude; depois desempenho.

## Sequência

1. Ambiente e compilação
2. Medição de desempenho
3. Modelo Fork–Join e `parallel`
4. `parallel for`
5. `schedule`
6. Escopo de dados
7. Race conditions
8. `critical`, `atomic` e `reduction`
9. `barrier`, `nowait` e `ordered`
10. `single`
11. `master` / `masked`
12. `sections`
13. `collapse`
14. OpenMP SIMD
15. `parallel for simd`
16. Tasks
17. Granularidade e recursão
18. Dependências entre tasks
19. `taskloop`
20. Locks
21. Afinidade
22. False sharing e localidade
23. Paralelismo aninhado
24–28. Estudos de caso
29. Automação de experimentos
30–39. Exercícios, projetos e checklist

## Google Colab e notação matemática

As células OpenMP usam C compilado com GCC diretamente no notebook.

Exemplo:

```c
%%writefile programa.c
...
```

e:

```bash
!gcc -O3 -march=native -fopenmp programa.c -o programa
```

O Colab pode oferecer poucas CPUs. Portanto, os exemplos funcionam, mas o *speedup* máximo depende da máquina atribuída.

### Matemática no Colab

Este notebook usa apenas a sintaxe MathJax mais compatível:

Inline:

`$S_p = T_1/T_p$`

Destacada:

`$$S_p = T_1/T_p$$`

As equações destacadas usam exclusivamente pares de `$$`, para maximizar a compatibilidade com o Google Colab.


# Antes de começar — o modelo mental de OpenMP

OpenMP é voltado principalmente a máquinas de **memória compartilhada**. Isso significa que todas as threads do processo conseguem acessar os mesmos vetores, matrizes e estruturas alocados na memória do programa.

O programador não cria threads manualmente para cada unidade de trabalho. Em vez disso, adiciona diretivas que descrevem **onde existe paralelismo** e **como os dados devem ser tratados**. O compilador e o runtime OpenMP transformam essas diretivas em criação de equipes, distribuição de iterações, sincronizações e agendamento de tasks.

Ao estudar qualquer diretiva deste notebook, use sempre quatro perguntas:

1. **Quem executa esta região?** Uma thread, todas as threads ou uma task?
2. **Como o trabalho é distribuído?** Estaticamente, dinamicamente ou pelo runtime de tasks?
3. **Quais dados são compartilhados e quais são privados?**
4. **Onde as threads precisam esperar umas pelas outras?**

Essas quatro perguntas resolvem grande parte das dúvidas de corretude em OpenMP.

## Estrutura conceitual

```text
Programa sequencial
        |
        v
  parallel
  cria equipe
        |
        +-----------------------------+
        |             |               |
     thread 0       thread 1        thread N
        |             |               |
        +----- trabalho paralelo ------+
                      |
             sincronização/join
                      |
                      v
             programa sequencial
```

Nos próximos tópicos, novos construtos não substituem esse modelo: eles especificam com mais precisão **o que cada thread faz dentro dele**.


# 1. Ambiente e compilação

OpenMP é uma API para programação paralela em arquiteturas de memória compartilhada.

Usaremos:

```text
-fopenmp
```

e, para desempenho:

```text
-O3 -march=native
```

### Como esta parte funciona

OpenMP não é uma linguagem separada: ele estende C/C++ por meio de **diretivas de compilação**, funções de runtime e variáveis de ambiente. Uma diretiva como `#pragma omp parallel` é reconhecida pelo compilador quando o suporte a OpenMP está habilitado. No GCC isso ocorre com `-fopenmp`.

Ao compilar com `-fopenmp`, duas coisas importantes acontecem: o compilador interpreta as diretivas `#pragma omp ...` e o executável é ligado à biblioteca de runtime OpenMP. Essa biblioteca é responsável por criar e gerenciar threads, implementar barreiras, escalonamento, tasks, locks e outras operações em tempo de execução.

A macro `_OPENMP` informa a versão da especificação que o compilador declara suportar. Já funções como `omp_get_max_threads()` e `omp_get_num_procs()` consultam o runtime e ajudam a entender os recursos disponíveis.

**O que observar no exemplo:** o número de CPUs visíveis ao programa não precisa coincidir com o número físico de cores da máquina. Em ambientes virtualizados, containers e Google Colab, o processo pode enxergar apenas uma fração do hardware.

In [ ]:
import os
import re
import sys
import platform
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("CPUs lógicas:", os.cpu_count())

gcc = subprocess.run(
    ["bash", "-lc", "gcc --version | head -n 1"],
    capture_output=True, text=True
)
print(gcc.stdout.strip())

In [ ]:
%%writefile teste_openmp.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    printf("_OPENMP = %d\n", _OPENMP);
    printf("Max threads = %d\n", omp_get_max_threads());
    printf("Num processors = %d\n", omp_get_num_procs());

    #pragma omp parallel
    {
        #pragma omp single
        printf("Threads na região = %d\n", omp_get_num_threads());
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp teste_openmp.c -o teste_openmp
!OMP_NUM_THREADS=4 ./teste_openmp

### Exercícios 1

**1.1.** Execute `lscpu` e identifique CPUs lógicas, sockets, cores por socket e threads por core.

**1.2.** Teste `OMP_NUM_THREADS=1,2,4,8,16`. O runtime sempre cria a quantidade pedida?

**1.3.** Imprima os valores de `omp_get_thread_limit()` e `omp_get_dynamic()`.

**1.4.** Use `sched_getcpu()` para mostrar em qual CPU cada thread executa.

**1.5.** Explique por que CPUs lógicas e cores físicos não são necessariamente a mesma coisa.

**1.6.** Descubra qual versão de OpenMP o valor de `_OPENMP` representa.

# 2. Medição de desempenho

Para $p$ threads:

$$
S_p = \frac{T_1}{T_p}
$$

A eficiência é:

$$
E_p = \frac{S_p}{p}
$$

Pela Lei de Amdahl, se $f$ é a fração serial:

$$
S_p = \frac{1}{f + \frac{1-f}{p}}
$$

e:

$$
S_{\infty} = \frac{1}{f}
$$

### Como medir corretamente

Paralelizar um código só é útil se conseguirmos avaliar o resultado de forma objetiva. O tempo com uma thread, $T_1$, funciona como referência para calcular o *speedup*:

$$
S_p = \frac{T_1}{T_p}
$$

Se quatro threads executam o programa duas vezes mais rápido, então $S_4=2$. A eficiência indica quanto do paralelismo teoricamente disponível está sendo aproveitado:

$$
E_p = \frac{S_p}{p}
$$

Na prática, speedup perfeitamente linear é raro. Criação e sincronização de threads possuem custo, partes do código continuam sequenciais, threads podem disputar largura de banda de memória e a carga pode não ser distribuída igualmente.

A Lei de Amdahl ajuda a entender um limite importante: uma pequena parte serial pode limitar fortemente o ganho máximo. Por isso, medir apenas uma região isolada pode produzir uma impressão diferente do desempenho da aplicação completa.

**Boa prática:** execute várias repetições, descarte ou separe aquecimento quando necessário e reporte dispersão. Um único tempo não caracteriza de forma confiável o comportamento de um programa paralelo.

In [ ]:
def resumo(tempos):
    x = np.asarray(tempos, dtype=float)
    return {
        "media": x.mean(),
        "mediana": np.median(x),
        "desvio": x.std(ddof=1) if len(x) > 1 else 0.0,
        "min": x.min(),
        "max": x.max(),
    }

def amdahl(p, f):
    p = np.asarray(p, dtype=float)
    return 1.0 / (f + (1.0-f)/p)

p = np.arange(1, 65)
for f in [0.01, 0.05, 0.10, 0.20]:
    plt.plot(p, amdahl(p, f), label=f"f={f:.0%}")

plt.xlabel("Threads")
plt.ylabel("Speedup")
plt.title("Lei de Amdahl")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercícios 2

**2.1.** Calcule o speedup máximo para frações seriais de 1%, 5%, 10% e 25%.

**2.2.** Se $T_1=12$ s e $T_8=2$ s, calcule $S_8$ e $E_8$.

**2.3.** Um código obteve speedup 6,2 em 8 threads. Que outras informações são necessárias para interpretar esse resultado?

**2.4.** Explique por que uma única execução é insuficiente.

**2.5.** Compare strong scaling e weak scaling.

**2.6.** Por que uma aceleração superlinear pode eventualmente aparecer?

# 3. Modelo Fork–Join e `parallel`

Uma execução OpenMP começa com uma thread inicial.

Ao encontrar:

```c
#pragma omp parallel
```

uma equipe é criada. Cada thread executa o bloco.

### Como `parallel` funciona

OpenMP usa principalmente o modelo **Fork–Join**. O programa começa com uma única thread, chamada *initial thread*. Quando essa thread encontra uma região `parallel`, o runtime forma uma **equipe de threads**. Todas as threads da equipe executam o bloco associado à diretiva.

Cada thread recebe um identificador entre `0` e `num_threads-1`. A thread 0 da equipe é chamada de *primary thread*. As threads compartilham o espaço de endereçamento do processo, mas possuem pilhas e estados de execução próprios.

Ao final de uma região `parallel`, existe uma sincronização implícita: todas as threads precisam concluir o bloco antes de a execução prosseguir fora dele. Depois ocorre o *join* e a execução volta a uma única thread.

**Ponto importante:** criar uma região paralela não significa automaticamente dividir o trabalho. Se o bloco contém `printf("Olá")`, todas as threads executarão o mesmo `printf`. A divisão de iterações é introduzida posteriormente por construtos como `for`.

In [ ]:
%%writefile omp_parallel.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    printf("Antes da região paralela\n");

    #pragma omp parallel
    {
        int id = omp_get_thread_num();
        int n  = omp_get_num_threads();
        printf("Olá da thread %d de %d\n", id, n);
    }

    printf("Depois da região paralela\n");
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_parallel.c -o omp_parallel
!OMP_NUM_THREADS=4 ./omp_parallel

## 3.1 `num_threads`

### O que `num_threads` controla

A cláusula `num_threads(n)` solicita que uma região paralela use `n` threads. Ela afeta aquela região específica, enquanto `OMP_NUM_THREADS` define uma configuração de runtime mais geral.

O valor é uma **solicitação**, não uma garantia absoluta. O runtime pode estar sujeito a limites do sistema, configurações de paralelismo dinâmico ou restrições de recursos.

Use `num_threads` quando o número de threads fizer parte explícita do experimento ou quando uma região específica precisar de uma configuração diferente. Em aplicações comuns, costuma ser mais conveniente deixar o número de threads configurável por variável de ambiente.

In [ ]:
%%writefile omp_num_threads.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel num_threads(3)
    {
        printf("thread %d / %d\n",
               omp_get_thread_num(),
               omp_get_num_threads());
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_num_threads.c -o omp_num_threads
!OMP_NUM_THREADS=8 ./omp_num_threads

### Exercícios 3 — `parallel`

**3.1.** Execute o exemplo dez vezes. A ordem das mensagens é determinística?

**3.2.** Faça somente a thread 0 imprimir o número total de threads.

**3.3.** Faça cada thread calcular `id*id`.

**3.4.** Divida manualmente um vetor entre threads usando `omp_get_thread_num()` e `omp_get_num_threads()`.

**3.5.** Inicialize um vetor de 100 elementos sem usar `omp for`.

**3.6.** Compare a divisão manual com o mecanismo `omp for`.

**3.7.** Abra e feche uma região `parallel` muitas vezes e compare com uma única região paralela contendo mais trabalho.

**3.8.** Explique o custo conceitual de fork/join.

# 4. `parallel for`

Quando as iterações são independentes:

```c
#pragma omp parallel for
for (...) {
    ...
}
```

distribui as iterações entre as threads.

### Como `parallel for` divide o trabalho

`parallel for` combina dois conceitos: criação de uma equipe de threads e distribuição das iterações de um laço entre essa equipe.

Em:

```c
#pragma omp parallel for
for (long i=0; i<N; i++)
    c[i] = a[i] + b[i];
```

cada valor de `i` é executado por **uma única thread**. Se as iterações são independentes, várias podem ocorrer ao mesmo tempo.

A condição fundamental é que uma iteração não dependa de resultados produzidos por outra. Por exemplo, `c[i]=a[i]+b[i]` é naturalmente paralelo porque cada iteração lê e escreve posições independentes. Em contraste, `a[i]=a[i-1]+1` possui uma dependência entre iterações e não pode ser paralelizado diretamente dessa forma.

No final do `for` existe uma barreira implícita, salvo quando o construto permitir `nowait`.

**Antes de usar `parallel for`, pergunte:** cada iteração poderia ser executada em qualquer ordem sem alterar a semântica do algoritmo?

In [ ]:
%%writefile omp_for_vector.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    long n = (argc > 1) ? atol(argv[1]) : 10000000L;

    double *a = malloc(n*sizeof(double));
    double *b = malloc(n*sizeof(double));
    double *c = malloc(n*sizeof(double));

    for (long i=0; i<n; i++) {
        a[i] = 1.0;
        b[i] = 2.0;
    }

    double t0 = omp_get_wtime();

    #pragma omp parallel for
    for (long i=0; i<n; i++)
        c[i] = a[i] + b[i];

    double t1 = omp_get_wtime();

    long erros = 0;
    for (long i=0; i<n; i++)
        if (c[i] != 3.0) erros++;

    printf("threads=%d n=%ld tempo=%.6f erros=%ld\n",
           omp_get_max_threads(), n, t1-t0, erros);

    free(a); free(b); free(c);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_for_vector.c -o omp_for_vector
for t in 1 2 4 8; do
    OMP_NUM_THREADS=$t ./omp_for_vector 5000000
done

## 4.1 Aplicação: SAXPY

$$
y_i = a x_i + y_i
$$

### Por que SAXPY é um exemplo importante

SAXPY aplica a mesma expressão a todos os elementos:

$$
y_i = a x_i + y_i
$$

As iterações são independentes, então o paralelismo é simples. Ao mesmo tempo, cada elemento exige poucas operações aritméticas e acessa vários dados na memória. Isso torna SAXPY um exemplo clássico para mostrar que **ter muitas operações independentes não garante speedup linear**.

À medida que aumentamos o número de threads, elas passam a compartilhar a largura de banda disponível para acessar a memória principal. Em algum ponto, adicionar threads pode deixar de reduzir o tempo porque o gargalo deixa de ser capacidade computacional e passa a ser movimentação de dados.

Por isso, use este exemplo não apenas para observar `parallel for`, mas para começar a distinguir aplicações **compute-bound** e **memory-bound**.

In [ ]:
%%writefile omp_saxpy.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    long n = (argc > 1) ? atol(argv[1]) : 10000000L;
    float alpha = 2.5f;

    float *x = malloc(n*sizeof(float));
    float *y = malloc(n*sizeof(float));

    #pragma omp parallel for
    for (long i=0; i<n; i++) {
        x[i] = 1.0f;
        y[i] = 2.0f;
    }

    double t0 = omp_get_wtime();

    #pragma omp parallel for
    for (long i=0; i<n; i++)
        y[i] = alpha*x[i] + y[i];

    double t1 = omp_get_wtime();

    printf("threads=%d tempo=%.6f y0=%.2f\n",
           omp_get_max_threads(), t1-t0, y[0]);

    free(x); free(y);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_saxpy.c -o omp_saxpy
!OMP_NUM_THREADS=4 ./omp_saxpy 5000000

### Exercícios 4 — `parallel for`

**4.1.** Converta SAXPY para `double`.

**4.2.** Calcule $y_i=\sqrt{x_i}+\sin(x_i)$.

**4.3.** Implemente threshold de uma imagem em tons de cinza.

**4.4.** Calcule a distância de cada ponto 2D à origem.

**4.5.** Teste tamanhos de $10$ até $10^8$. Quando a paralelização passa a compensar?

**4.6.** Gere curvas de tempo, speedup e eficiência.

**4.7.** Implemente conversão Celsius → Fahrenheit para milhões de amostras.

**4.8.** Calcule a norma L1 de dois vetores sem usar ainda `reduction`.

**4.9.** Implemente uma versão matriz-vetor.

**4.10.** Explique quais laços não podem ser paralelizados apenas porque “possuem muitas iterações”.

# 5. `schedule` — imediatamente após `parallel for`

A cláusula:

```c
schedule(tipo, chunk)
```

controla a distribuição de iterações.

- `static`: distribuição pré-determinada;
- `dynamic`: as threads requisitam novos blocos;
- `guided`: começa com blocos maiores e reduz;
- `runtime`: usa `OMP_SCHEDULE`.

A escolha envolve:

- balanceamento;
- overhead;
- granularidade;
- localidade de memória.

### O que o `schedule` decide

Depois de decidir que um laço pode ser paralelizado, ainda precisamos decidir **quais iterações cada thread executará**. Essa é a função de `schedule`.

Com `static`, o espaço de iteração é dividido antecipadamente. O overhead é pequeno e a localidade costuma ser boa, por isso é uma excelente escolha para trabalho regular.

Com `dynamic`, cada thread solicita novos blocos de iterações ao runtime quando termina o bloco atual. Isso melhora o balanceamento quando o custo das iterações varia, mas adiciona overhead de gerenciamento.

`guided` também distribui trabalho dinamicamente, porém começa com blocos maiores e reduz o tamanho ao longo da execução. A intenção é combinar balanceamento com menor overhead.

O parâmetro `chunk` controla a granularidade da distribuição. Chunks muito pequenos aumentam a flexibilidade, mas também aumentam a quantidade de decisões do runtime.

**Ideia central:** `schedule` é um compromisso entre **balanceamento de carga, overhead e localidade de memória**.

In [ ]:
%%writefile omp_schedule_map.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel for schedule(static,2)
    for (int i=0; i<16; i++) {
        #pragma omp critical
        printf("iteracao=%2d thread=%d\n",
               i, omp_get_thread_num());
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_schedule_map.c -o omp_schedule_map
!OMP_NUM_THREADS=4 ./omp_schedule_map

## 5.1 Aplicação irregular

### Por que a carga irregular muda o resultado

Neste exemplo, a quantidade de trabalho executada pela iteração `i` varia. Isso significa que dividir apenas o número de iterações igualmente não garante dividir o **tempo de trabalho** igualmente.

Se uma thread recebe várias iterações caras e outra recebe várias baratas, a segunda termina antes e permanece ociosa até que a primeira alcance a barreira final. Esse tempo ocioso é uma forma de desbalanceamento.

`dynamic` e `guided` tentam reduzir esse efeito fazendo a distribuição durante a execução. O preço é um overhead maior no runtime. Portanto, o melhor schedule depende da relação entre **custo de cada iteração** e **custo de redistribuir trabalho**.

In [ ]:
%%writefile omp_schedule_irregular.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

static double trabalho(int i) {
    int reps = 1000 + (i % 1000)*50;
    double x = 0.0;

    for (int k=0; k<reps; k++)
        x += sin(0.00001*(k+i));

    return x;
}

int main(int argc, char **argv) {
    int n = (argc > 1) ? atoi(argv[1]) : 8000;
    double soma = 0.0;

    double t0 = omp_get_wtime();

    #pragma omp parallel for schedule(runtime) reduction(+:soma)
    for (int i=0; i<n; i++)
        soma += trabalho(i);

    double t1 = omp_get_wtime();

    printf("tempo=%.6f soma=%.8f\n", t1-t0, soma);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_schedule_irregular.c -lm -o omp_schedule_irregular

for sched in static static,16 dynamic,1 dynamic,16 dynamic,128 guided; do
    echo "---- $sched"
    OMP_NUM_THREADS=4 OMP_SCHEDULE=$sched ./omp_schedule_irregular 5000
done

## 5.2 Aplicação: números primos

### O que torna a busca de primos interessante

O teste de primalidade executa quantidades diferentes de operações para números diferentes. Alguns compostos encontram um divisor rapidamente, enquanto números primos exigem testar muitos candidatos até aproximadamente $\sqrt{n}$.

Isso produz uma carga irregular real, em vez de artificial. O exercício permite observar que uma política ótima para um laço uniforme pode não ser ótima para um laço cujo custo depende dos dados.

Além do schedule, observe que uma redução é usada para acumular a quantidade de primos. Assim, o exemplo combina dois problemas diferentes: **distribuição do trabalho** e **combinação segura de resultados**.

In [ ]:
%%writefile omp_primes_schedule.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

int primo(long n) {
    if (n < 2) return 0;
    if (n == 2) return 1;
    if (n % 2 == 0) return 0;

    long limite = (long)sqrt((double)n);
    for (long d=3; d<=limite; d+=2)
        if (n % d == 0) return 0;

    return 1;
}

int main(int argc, char **argv) {
    long n = (argc > 1) ? atol(argv[1]) : 500000L;
    long qtd = 0;

    double t0 = omp_get_wtime();

    #pragma omp parallel for schedule(runtime) reduction(+:qtd)
    for (long i=2; i<=n; i++)
        qtd += primo(i);

    double t1 = omp_get_wtime();

    printf("N=%ld primos=%ld tempo=%.6f\n", n, qtd, t1-t0);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_primes_schedule.c -lm -o omp_primes_schedule
for sched in static dynamic,1 dynamic,64 guided; do
    echo "---- $sched"
    OMP_NUM_THREADS=4 OMP_SCHEDULE=$sched ./omp_primes_schedule 300000
done

### Exercícios 5 — `schedule`

**5.1.** Para 20 iterações e 4 threads, registre `iteração → thread` em `static`, `static,1`, `static,4`, `dynamic,1` e `dynamic,4`.

**5.2.** Teste chunks 1, 2, 4, 8, 16, 32, 64, 128 e 512. Gere `chunk × tempo`.

**5.3.** Torne a carga completamente regular. Qual política tende a vencer?

**5.4.** Teste primos com vários limites.

**5.5.** Paralelize um espaço triangular de iteração e estude o balanceamento.

**5.6.** Antecipe qual política deve funcionar bem no Mandelbrot; depois valide.

**5.7.** Compare `static` e `dynamic,1` em um vetor grande e regular. Discuta localidade.

**5.8.** Use `schedule(runtime)` para automatizar testes sem recompilar.

**5.9.** Meça overhead de `dynamic,1` quando cada iteração faz quase nada.

**5.10.** Explique em até dez linhas quando usar `static`, `dynamic` e `guided`.

# 6. Escopo de dados

Cláusulas fundamentais:

- `shared(x)`
- `private(x)`
- `firstprivate(x)`
- `lastprivate(x)`
- `default(none)`

`default(none)` é especialmente útil para tornar o escopo das variáveis explícito.

### Memória compartilhada não significa que toda variável deve ser compartilhada

As threads OpenMP vivem no mesmo processo e, portanto, podem acessar a mesma memória. Porém, cada variável precisa ter uma semântica adequada.

Uma variável `shared` representa um único objeto visível por todas as threads. Uma variável `private` cria uma instância independente para cada thread, sem preservar necessariamente o valor anterior à região. `firstprivate` também cria uma cópia por thread, mas inicializa essa cópia com o valor existente antes da região.

`lastprivate` resolve outro problema: ao final de um laço paralelo, queremos preservar o valor correspondente à **última iteração lógica**, independentemente de qual thread a executou.

`default(none)` é uma ferramenta de engenharia de software muito útil. Ela força o programador a declarar explicitamente o compartilhamento das variáveis relevantes, reduzindo erros silenciosos.

**Pergunta a fazer para cada variável:** várias threads precisam enxergar o mesmo objeto ou cada thread deveria possuir sua própria cópia?

In [ ]:
%%writefile omp_private.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    int x = 10;

    #pragma omp parallel private(x)
    {
        x = 100 + omp_get_thread_num();
        printf("thread %d: x=%d\n", omp_get_thread_num(), x);
    }

    printf("fora: x=%d\n", x);
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_private.c -o omp_private
!OMP_NUM_THREADS=4 ./omp_private

In [ ]:
%%writefile omp_first_last.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    int x = 10;
    int ultimo = -1;

    #pragma omp parallel firstprivate(x)
    {
        x += omp_get_thread_num();
        printf("firstprivate thread=%d x=%d\n",
               omp_get_thread_num(), x);
    }

    #pragma omp parallel for lastprivate(ultimo)
    for (int i=0; i<20; i++)
        ultimo = i*i;

    printf("x original=%d ultimo=%d\n", x, ultimo);
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_first_last.c -o omp_first_last
!OMP_NUM_THREADS=4 ./omp_first_last

### Exercícios 6 — escopo

**6.1.** Compare `private` e `firstprivate`.

**6.2.** O valor inicial de uma variável `private` é definido?

**6.3.** Faça cada thread manter um contador local.

**6.4.** Crie um exemplo útil para `lastprivate`.

**6.5.** Reescreva um exemplo anterior com `default(none)`.

**6.6.** Produza intencionalmente um erro de compilação omitindo uma variável com `default(none)`.

**6.7.** Classifique variáveis de uma simulação de partículas como `shared`, `private` ou `firstprivate`.

**6.8.** Explique o escopo padrão do índice de um `omp for`.

# 7. Condições de corrida

Uma race condition ocorre quando acessos concorrentes a dados compartilhados fazem o resultado depender da ordem de execução.

`contador++` envolve leitura, incremento e escrita.

### O que é uma race condition

Uma condição de corrida ocorre quando várias threads acessam o mesmo estado compartilhado, pelo menos uma delas modifica esse estado e a sincronização necessária não existe.

O exemplo clássico é:

```c
contador++;
```

Essa linha parece uma operação única no código-fonte, mas normalmente envolve ler `contador`, calcular o novo valor e escrever o resultado. Duas threads podem ler o mesmo valor antigo e ambas escrever o mesmo incremento, fazendo uma atualização desaparecer.

O comportamento é especialmente perigoso porque o erro é **não determinístico**. O programa pode funcionar em uma execução e falhar em outra, ou mudar quando adicionamos `printf`, alteramos otimização ou executamos em outra máquina.

A solução não é simplesmente “colocar um lock em tudo”. Primeiro identifique qual estado realmente precisa ser compartilhado e procure a abstração de sincronização mais específica para o problema.

In [ ]:
%%writefile omp_race.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    long soma = 0;
    const long N = 10000000L;

    #pragma omp parallel for
    for (long i=0; i<N; i++)
        soma++;

    printf("esperado=%ld obtido=%ld\n", N, soma);
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_race.c -o omp_race
for r in 1 2 3 4 5; do
    OMP_NUM_THREADS=4 ./omp_race
done

### Exercícios 7 — races

Analise:

**7.1**
```c
#pragma omp parallel for
for (int i=0; i<N; i++)
    a[i] = i;
```

**7.2**
```c
int x = 0;
#pragma omp parallel for
for (int i=0; i<N; i++)
    x = i;
```

**7.3**
```c
#pragma omp parallel for
for (int i=1; i<N; i++)
    a[i] = a[i-1] + 1;
```

**7.4**
```c
#pragma omp parallel for
for (int i=0; i<N; i++)
    c[i] = a[i] + b[i];
```

**7.5.** Analise `hist[pixel]++` com histograma compartilhado.

**7.6.** Analise atualizações de partículas na mesma célula de uma grade.

**7.7.** Explique por que um programa com race pode ocasionalmente produzir o resultado esperado.

**7.8.** Crie um exemplo em que a race não altera o resultado final e explique por que isso ainda é programação incorreta.

# 8. `critical`, `atomic` e `reduction`

- `critical`: uma thread por vez executa um bloco;
- `atomic`: protege uma atualização simples;
- `reduction`: usa acumuladores privados e combina os resultados.

Para somas, `reduction` costuma ser a abstração mais adequada.

### Três formas diferentes de resolver conflitos

`critical` garante exclusão mútua para um bloco arbitrário. É flexível, mas somente uma thread executa aquela região por vez. Se a região crítica for frequente ou longa, ela pode se tornar um gargalo serial.

`atomic` é mais restrito: protege operações específicas sobre uma posição de memória. Como o runtime/compilador conhece melhor a operação, a implementação geralmente é mais leve do que uma região crítica genérica.

`reduction` é diferente. Em vez de fazer todas as threads atualizarem continuamente uma mesma variável, OpenMP cria valores parciais privados e combina esses valores no final. Para soma, produto, mínimo, máximo e operações similares, isso reduz drasticamente a contenção.

**Regra prática:** use a abstração de maior nível que expressa corretamente o algoritmo. Para uma soma global, prefira `reduction`; para uma atualização simples compartilhada, considere `atomic`; para operações compostas que realmente precisam ser indivisíveis, use `critical`.

In [ ]:
%%writefile omp_sync_compare.c
#include <stdio.h>
#include <string.h>
#include <omp.h>

long teste_critical(long N) {
    long soma = 0;
    #pragma omp parallel for
    for (long i=0; i<N; i++) {
        #pragma omp critical
        soma++;
    }
    return soma;
}

long teste_atomic(long N) {
    long soma = 0;
    #pragma omp parallel for
    for (long i=0; i<N; i++) {
        #pragma omp atomic
        soma++;
    }
    return soma;
}

long teste_reduction(long N) {
    long soma = 0;
    #pragma omp parallel for reduction(+:soma)
    for (long i=0; i<N; i++)
        soma++;
    return soma;
}

int main(int argc, char **argv) {
    const long N = 3000000L;
    const char *modo = (argc > 1) ? argv[1] : "reduction";

    double t0 = omp_get_wtime();
    long r;

    if (!strcmp(modo, "critical"))
        r = teste_critical(N);
    else if (!strcmp(modo, "atomic"))
        r = teste_atomic(N);
    else
        r = teste_reduction(N);

    double t1 = omp_get_wtime();

    printf("%s resultado=%ld tempo=%.6f\n",
           modo, r, t1-t0);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_sync_compare.c -o omp_sync_compare
for modo in critical atomic reduction; do
    OMP_NUM_THREADS=4 ./omp_sync_compare $modo
done

## 8.1 Produto escalar com `reduction`

$$
d = \sum_{i=0}^{N-1} a_i b_i
$$

### Como a redução aparece em aplicações reais

No produto escalar, cada iteração calcula uma contribuição independente `a[i]*b[i]`. O único conflito está na soma dessas contribuições.

Com `reduction(+:dot)`, cada thread acumula uma soma parcial. No final do construto, o runtime combina essas somas em `dot`. Isso permite que a maior parte do cálculo aconteça sem disputa pela mesma variável.

Em ponto flutuante, a ordem das adições pode mudar entre configurações paralelas. Como a aritmética IEEE 754 não é associativa em precisão finita, pequenas diferenças nos últimos dígitos são normais e não constituem necessariamente um erro de paralelização.

In [ ]:
%%writefile omp_dot.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    long N = (argc > 1) ? atol(argv[1]) : 5000000L;

    double *a = malloc(N*sizeof(double));
    double *b = malloc(N*sizeof(double));

    #pragma omp parallel for
    for (long i=0; i<N; i++) {
        a[i] = 1.0/(i+1);
        b[i] = 2.0;
    }

    double dot = 0.0;
    double t0 = omp_get_wtime();

    #pragma omp parallel for reduction(+:dot)
    for (long i=0; i<N; i++)
        dot += a[i]*b[i];

    double t1 = omp_get_wtime();

    printf("dot=%.12f tempo=%.6f\n", dot, t1-t0);

    free(a); free(b);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_dot.c -o omp_dot
!OMP_NUM_THREADS=4 ./omp_dot 5000000

### Exercícios 8 — sincronização

**8.1.** Compare `critical`, `atomic` e `reduction` com 1, 2, 4 e 8 threads.

**8.2.** Explique por que `critical` pode serializar o programa.

**8.3.** Dê um caso em que `critical` é necessário e `atomic` não basta.

**8.4.** Some 100 milhões de elementos com `reduction`.

**8.5.** Encontre mínimo e máximo de um vetor.

**8.6.** Calcule:

$$
||x||_2 = \sqrt{\sum_i x_i^2}
$$

**8.7.** Conte valores acima de um limiar.

**8.8.** Calcule média e variância.

**8.9.** Compare histograma com `critical`, `atomic` e privatização.

**8.10.** Compare últimos dígitos de uma soma `double` com diferentes números de threads.

**8.11.** Implemente redução manual por thread e compare com a cláusula `reduction`.

**8.12.** Investigue `reduction` com operações `min`, `max`, `&`, `|` e `^`.

# 9. `barrier`, `nowait` e `ordered`

Uma barreira força todas as threads a alcançar um ponto antes de prosseguir.

```c
#pragma omp barrier
```

`nowait` remove certas barreiras implícitas.

Isso só é correto quando não existe dependência entre as fases.

### Sincronização entre fases

Uma `barrier` representa um ponto de encontro. Nenhuma thread pode passar desse ponto até que todas as threads da equipe tenham chegado.

Muitos construtos OpenMP já possuem barreiras implícitas. Isso simplifica a corretude, mas pode introduzir espera desnecessária. `nowait` remove determinadas barreiras quando o programador sabe que a próxima fase não depende da conclusão de todas as threads.

`ordered` resolve outro tipo de problema: queremos executar o trabalho em paralelo, mas uma pequena parte precisa obedecer à ordem original das iterações. Preservar ordem reduz a liberdade do runtime e normalmente deve ser limitado à menor região possível.

**Cuidado:** remover uma barreira não é uma otimização puramente de desempenho; muda as garantias de sincronização do programa.

In [ ]:
%%writefile omp_nowait.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(void) {
    const long N = 5000000L;
    double *a = malloc(N*sizeof(double));
    double *b = malloc(N*sizeof(double));

    double t0 = omp_get_wtime();

    #pragma omp parallel
    {
        #pragma omp for nowait
        for (long i=0; i<N; i++)
            a[i] = (double)i;

        #pragma omp for
        for (long i=0; i<N; i++)
            b[i] = 2.0*i;
    }

    double t1 = omp_get_wtime();

    printf("tempo=%.6f a=%f b=%f\n",
           t1-t0, a[N-1], b[N-1]);

    free(a); free(b);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_nowait.c -o omp_nowait
!OMP_NUM_THREADS=4 ./omp_nowait

In [ ]:
%%writefile omp_ordered.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel for ordered schedule(dynamic,1)
    for (int i=0; i<12; i++) {
        int tid = omp_get_thread_num();

        #pragma omp ordered
        printf("iteracao=%d thread=%d\n", i, tid);
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_ordered.c -o omp_ordered
!OMP_NUM_THREADS=4 ./omp_ordered

### Exercícios 9

**9.1.** Remova `nowait` e compare o tempo.

**9.2.** Crie dois laços em que o segundo depende do primeiro. Demonstre por que `nowait` pode ser incorreto.

**9.3.** Faça cada thread executar fase A, `barrier`, fase B.

**9.4.** Explique o custo de barreiras excessivas.

**9.5.** Em um pipeline de três fases, identifique barreiras realmente necessárias.

**9.6.** Use `ordered` para produzir saída em ordem.

**9.7.** Compare `ordered` com armazenar os resultados e imprimir depois.

**9.8.** Meça o custo de uma barreira em um microbenchmark.

# 10. `single`

`single` faz um bloco ser executado por **uma thread qualquer**.

Por padrão há barreira implícita ao final, salvo com `nowait`.

### Uma operação executada uma única vez

Dentro de uma região `parallel`, todas as threads normalmente executam o mesmo bloco. `single` indica que uma determinada região deve ser executada por apenas **uma thread da equipe**, escolhida pelo runtime.

Isso é útil para inicializações, leitura de metadados, geração de tasks ou qualquer trabalho que só precisa ocorrer uma vez enquanto a equipe paralela já existe.

Por padrão, existe uma barreira no final de `single`. Portanto, as demais threads aguardam a conclusão da operação antes de prosseguir. Se essa garantia não for necessária, `nowait` pode removê-la.

Uma diferença essencial para `master`/`masked` é que `single` não exige que a thread escolhida seja a thread primária.

In [ ]:
%%writefile omp_single.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        printf("thread %d chegou\n", omp_get_thread_num());

        #pragma omp single
        {
            printf("SINGLE executado pela thread %d\n",
                   omp_get_thread_num());
        }

        printf("thread %d continuou\n", omp_get_thread_num());
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_single.c -o omp_single
!OMP_NUM_THREADS=4 ./omp_single

### Exercícios 10 — `single`

**10.1.** Execute várias vezes. A mesma thread executa `single`?

**10.2.** Adicione `nowait`.

**10.3.** Use `single` para inicializar estado compartilhado antes de todas as threads utilizarem o dado.

**10.4.** Simule leitura única de parâmetros de uma aplicação.

**10.5.** Explique por que `single` aparece frequentemente junto com criação de tasks.

**10.6.** Pesquise e teste `copyprivate`.

**10.7.** Compare semanticamente `single` com um `if (omp_get_thread_num()==0)`.

# 11. `master` e `masked`

`master` é comum em códigos OpenMP antigos:

```c
#pragma omp master
```

OpenMP moderno prefere:

```c
#pragma omp masked
```

Diferença em relação a `single`:

- `single`: uma thread qualquer;
- `master`/`masked`: thread selecionada pelo construto;
- não existe barreira implícita na saída de `master`/`masked`.

No OpenMP 5.2, `master` está depreciado, mas é importante conhecê-lo para leitura e manutenção de código.

### Quando queremos uma thread específica

`master` historicamente faz o bloco ser executado pela thread primária da equipe. OpenMP moderno introduziu `masked`, que generaliza esse comportamento e é a forma preferida em código novo.

Diferentemente de `single`, esses construtos não possuem uma barreira implícita no final. As outras threads podem continuar executando enquanto a thread selecionada ainda está dentro da região.

Isso pode ser útil para monitoramento, registro de estatísticas ou operações que devem ficar associadas à thread primária. Porém, se o restante do programa depender do resultado dessa operação, uma sincronização explícita pode ser necessária.

O notebook apresenta `master` porque ele ainda aparece com frequência em códigos científicos existentes, mesmo estando depreciado na especificação OpenMP 5.2 em favor de `masked`.

In [ ]:
%%writefile omp_master_masked.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        int tid = omp_get_thread_num();

        printf("thread %d iniciou\n", tid);

        #pragma omp master
        {
            printf("MASTER: thread %d\n", tid);
        }

        #pragma omp barrier

        #pragma omp masked
        {
            printf("MASKED: thread %d\n", tid);
        }
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_master_masked.c -o omp_master_masked
!OMP_NUM_THREADS=4 ./omp_master_masked

### Exercícios 11

**11.1.** Compare `single`, `master` e `masked`.

**11.2.** Coloque uma operação lenta dentro de `master`. As outras threads esperam?

**11.3.** Adicione `barrier` depois de `master`.

**11.4.** Faça somente a thread principal imprimir estatísticas.

**11.5.** Simule escrita de checkpoint por uma única thread.

**11.6.** Se suportado, teste `masked filter(1)`.

**11.7.** Explique quando `single` é semanticamente melhor que `master/masked`.

# 12. `sections`

`sections` expressa paralelismo funcional: blocos diferentes podem executar simultaneamente.

### Paralelismo funcional

`parallel for` divide diferentes iterações da **mesma operação**. `sections` resolve outro padrão: temos operações diferentes e independentes que podem executar simultaneamente.

Cada bloco `section` é executado uma única vez por alguma thread da equipe. Se existem três sections e oito threads, no máximo três threads terão trabalho proveniente dessas sections.

Isso torna `sections` apropriado para paralelismo funcional grosso, por exemplo executar três análises independentes sobre um conjunto de dados.

No final de `sections` existe normalmente uma barreira implícita. Assim, a fase seguinte só começa quando todas as sections terminarem, salvo uso apropriado de `nowait`.

In [ ]:
%%writefile omp_sections.c
#include <stdio.h>
#include <omp.h>

void etapa_a(void) {
    for (volatile long i=0; i<30000000L; i++);
    printf("A: thread %d\n", omp_get_thread_num());
}

void etapa_b(void) {
    for (volatile long i=0; i<20000000L; i++);
    printf("B: thread %d\n", omp_get_thread_num());
}

void etapa_c(void) {
    for (volatile long i=0; i<10000000L; i++);
    printf("C: thread %d\n", omp_get_thread_num());
}

int main(void) {
    #pragma omp parallel sections
    {
        #pragma omp section
        etapa_a();

        #pragma omp section
        etapa_b();

        #pragma omp section
        etapa_c();
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_sections.c -o omp_sections
!OMP_NUM_THREADS=4 ./omp_sections

### Exercícios 12 — `sections`

**12.1.** Execute com 1, 2, 3, 4 e 8 threads.

**12.2.** Faça uma section 10× mais lenta.

**12.3.** Calcule média, máximo e contagem em três sections independentes.

**12.4.** Simule leitura de três arquivos independentes.

**12.5.** Compare `sections` com três tasks.

**12.6.** Teste `nowait` em `sections`.

**12.7.** Implemente duas etapas independentes A/B seguidas de C, que depende das duas.

**12.8.** Dê um exemplo real em que `sections` é melhor representação do problema do que `parallel for`.

# 13. `collapse`

`collapse(2)` combina logicamente dois espaços de iteração aninhados.

### Expondo mais paralelismo em laços aninhados

Quando paralelizamos apenas o laço externo de uma estrutura aninhada, o número de unidades de trabalho pode ser pequeno. Se existem apenas duas linhas de uma matriz e oito threads, somente duas threads podem receber iterações do laço externo.

`collapse(2)` combina os espaços de iteração dos dois laços associados em um espaço lógico maior. Assim, pares `(i,j)` podem ser distribuídos entre mais threads.

Isso pode melhorar balanceamento e exposição de paralelismo, mas também pode alterar localidade. Em matrizes armazenadas por linhas, manter uma mesma thread processando uma linha inteira pode aproveitar melhor cache do que distribuir pequenos pedaços dessa linha.

Portanto, `collapse` deve ser avaliado experimentalmente: mais paralelismo disponível não significa automaticamente menor tempo.

In [ ]:
%%writefile omp_collapse.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    const int N = 4;
    const int M = 6;

    #pragma omp parallel for collapse(2) schedule(static)
    for (int i=0; i<N; i++) {
        for (int j=0; j<M; j++) {
            #pragma omp critical
            printf("(%d,%d) -> thread %d\n",
                   i, j, omp_get_thread_num());
        }
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_collapse.c -o omp_collapse
!OMP_NUM_THREADS=4 ./omp_collapse

### Exercícios 13 — `collapse`

**13.1.** Compare distribuição com e sem `collapse(2)`.

**13.2.** Use 2 linhas e muitas colunas.

**13.3.** Inicialize uma matriz 2D.

**13.4.** Some todos os elementos com `collapse(2)` + `reduction`.

**13.5.** Use `collapse(3)` em uma grade 3D.

**13.6.** Investigue restrições sobre os laços associados.

**13.7.** Compare localidade de memória com paralelização somente por linhas.

**13.8.** Explique por que melhor balanceamento não garante melhor desempenho.

# 14. OpenMP SIMD

Até aqui exploramos paralelismo entre **threads**. Processadores modernos também executam uma operação sobre vários elementos usando instruções vetoriais.

```c
#pragma omp simd
for (...) {
    ...
}
```

- `parallel for` → paralelismo entre threads;
- `simd` → paralelismo vetorial dentro de um core;
- `parallel for simd` → combina os dois níveis.

### Paralelismo dentro de um core

Threads OpenMP exploram múltiplos cores ou contextos de hardware. SIMD explora outra dimensão: uma única instrução vetorial processa vários elementos de dados.

Se uma CPU consegue operar, por exemplo, sobre oito valores `float` em um registrador vetorial, uma iteração vetorizada pode realizar várias operações simultaneamente dentro de uma mesma thread.

`#pragma omp simd` afirma que as iterações podem ser executadas de forma compatível com SIMD. Isso não deve ser usado para ignorar dependências reais. O programador continua responsável por preservar a semântica.

O ganho depende da arquitetura, do tipo de dado, das operações realizadas, do alinhamento e principalmente da existência de dependências. Compilar com relatórios de vetorização é fundamental para verificar o que o compilador realmente fez.

In [ ]:
%%writefile omp_simd.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    long N = (argc > 1) ? atol(argv[1]) : 10000000L;

    float *a = aligned_alloc(64, N*sizeof(float));
    float *b = aligned_alloc(64, N*sizeof(float));
    float *c = aligned_alloc(64, N*sizeof(float));

    for (long i=0; i<N; i++) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    double t0 = omp_get_wtime();

    #pragma omp simd
    for (long i=0; i<N; i++)
        c[i] = 3.0f*a[i] + b[i];

    double t1 = omp_get_wtime();

    printf("tempo=%.6f c=%f\n", t1-t0, c[N-1]);

    free(a); free(b); free(c);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp \
    -fopt-info-vec-optimized omp_simd.c -o omp_simd
!./omp_simd 5000000

## 14.1 Relatório de vetorização

Com GCC:

```text
-fopt-info-vec-optimized
-fopt-info-vec-missed
```

é possível verificar quais laços foram ou não vetorizados.

### Não suponha que um laço foi vetorizado

Compiladores modernos já tentam vetorizar laços automaticamente em níveis altos de otimização. Por isso, comparar apenas “com pragma” e “sem pragma” nem sempre demonstra SIMD: a versão sem pragma pode já estar vetorizada.

Os relatórios `-fopt-info-vec-optimized` e `-fopt-info-vec-missed` mostram quais laços foram vetorizados e por que outros não foram.

Ao analisar um relatório, procure razões como dependências, chamadas de função, acesso irregular ou impossibilidade de provar alinhamento. O objetivo é relacionar a transformação do compilador ao desempenho medido.

In [ ]:
!gcc -O3 -march=native -fopenmp \
    -fopt-info-vec-optimized \
    -fopt-info-vec-missed \
    omp_simd.c -o omp_simd_report 2> simd_report.txt

!sed -n '1,60p' simd_report.txt

## 14.2 SIMD com redução

### Reduções também podem ser vetorizadas

Uma redução aparentemente cria uma dependência porque cada iteração atualiza o mesmo acumulador. `reduction(+:soma)` informa ao compilador que a operação pode ser reorganizada usando acumuladores parciais.

Em SIMD, vários acumuladores vetoriais podem ser mantidos simultaneamente e combinados ao final. Isso reduz a cadeia de dependência e permite utilizar as unidades vetoriais.

Como em reduções entre threads, a ordem das operações de ponto flutuante pode mudar. Portanto, pequenas diferenças numéricas devem ser avaliadas com tolerância apropriada.

In [ ]:
%%writefile omp_simd_reduction.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(void) {
    const long N = 5000000L;
    double *a = aligned_alloc(64, N*sizeof(double));

    for (long i=0; i<N; i++)
        a[i] = 1.0/(i+1);

    double soma = 0.0;
    double t0 = omp_get_wtime();

    #pragma omp simd reduction(+:soma)
    for (long i=0; i<N; i++)
        soma += a[i];

    double t1 = omp_get_wtime();

    printf("soma=%.12f tempo=%.6f\n", soma, t1-t0);
    free(a);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp \
    -fopt-info-vec-optimized omp_simd_reduction.c -o omp_simd_reduction
!./omp_simd_reduction

## 14.3 Cláusulas SIMD avançadas

Exemplos:

```c
#pragma omp simd simdlen(8)
```

```c
#pragma omp simd safelen(4)
```

```c
#pragma omp simd aligned(a,b,c:64)
```

Essas cláusulas informam propriedades/restrições ao compilador.

> Não use `aligned(...)` se o alinhamento prometido não for verdadeiro.

### Informações adicionais para o compilador

`simdlen` sugere/define a quantidade preferida de iterações simultâneas no construto. `safelen` expressa uma distância máxima segura quando há restrições entre iterações.

A cláusula `aligned` informa que determinados ponteiros possuem um alinhamento conhecido. Isso pode permitir acessos vetoriais mais eficientes, mas é uma promessa do programador: declarar um alinhamento incorreto pode produzir comportamento inválido.

Essas cláusulas são recursos de otimização avançada. Primeiro obtenha uma versão correta e confirme a vetorização; depois investigue se fornecer mais informações ao compilador produz ganho real.

### Exercícios 14 — SIMD

**14.1.** Compare `-O0`, `-O2` e `-O3`.

**14.2.** Compare o mesmo laço sem diretiva e com `omp simd`.

**14.3.** Implemente SAXPY com `omp simd`.

**14.4.** Implemente produto escalar com `simd reduction`.

**14.5.** Vetorize:

$$
d_i = \sqrt{(x_i-a)^2 + (y_i-b)^2}
$$

**14.6.** Crie um laço com `if` e examine o relatório do compilador.

**14.7.** Analise por que este laço possui dependência:

```c
for (i=1; i<N; i++)
    a[i] = a[i-1] + b[i];
```

**14.8.** Compare `malloc` e `aligned_alloc`.

**14.9.** Compare `float` e `double`.

**14.10.** Calcule elementos/s e GB/s para uma transformação vetorial.

**14.11.** Gere assembly com `-S` e procure instruções vetoriais.

**14.12.** Teste `simdlen` quando suportado e verifique se a melhor configuração depende da arquitetura.

**14.13.** Investigue `safelen` em um exemplo com dependência de distância conhecida.

# 15. `parallel for simd`

O construto:

```c
#pragma omp parallel for simd
```

combina paralelismo entre threads com vetorização dentro de cada thread.

### Dois níveis de paralelismo

`parallel for simd` combina duas formas de paralelismo.

Primeiro, o espaço de iteração é dividido entre threads. Depois, dentro da parte atribuída a cada thread, o compilador pode agrupar iterações em instruções vetoriais SIMD.

Conceitualmente:

```text
dados
 ├─ thread 0 → vetores SIMD
 ├─ thread 1 → vetores SIMD
 ├─ thread 2 → vetores SIMD
 └─ thread 3 → vetores SIMD
```

Esse padrão é muito comum em kernels numéricos regulares. Contudo, quando o kernel já satura a largura de banda de memória com poucas threads, SIMD e threads adicionais podem não multiplicar seus ganhos de forma independente.

In [ ]:
%%writefile omp_parallel_simd.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    long N = (argc > 1) ? atol(argv[1]) : 10000000L;

    float *a = aligned_alloc(64, N*sizeof(float));
    float *b = aligned_alloc(64, N*sizeof(float));
    float *c = aligned_alloc(64, N*sizeof(float));

    #pragma omp parallel for simd
    for (long i=0; i<N; i++) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    double t0 = omp_get_wtime();

    #pragma omp parallel for simd
    for (long i=0; i<N; i++)
        c[i] = 3.0f*a[i] + b[i];

    double t1 = omp_get_wtime();

    printf("threads=%d tempo=%.6f c=%f\n",
           omp_get_max_threads(), t1-t0, c[N-1]);

    free(a); free(b); free(c);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp \
    -fopt-info-vec-optimized omp_parallel_simd.c -o omp_parallel_simd

for t in 1 2 4 8; do
    OMP_NUM_THREADS=$t ./omp_parallel_simd 5000000
done

### Exercícios 15 — threads + SIMD

**15.1.** Compare sequencial, `omp simd`, `omp parallel for` e `omp parallel for simd`.

**15.2.** Faça a comparação para vetores pequenos, médios e grandes.

**15.3.** Identifique quando a largura de banda de memória limita o ganho.

**15.4.** Faça uma soma com `parallel for simd reduction`.

**15.5.** Compare `sin`, `sqrt` e uma expressão aritmética simples.

**15.6.** Implemente:

$$
y_i = \frac{x_i-\mu}{\sigma}
$$

**15.7.** Classifique SAXPY como memory-bound ou compute-bound e justifique com dados.

**15.8.** Compare separadamente o ganho de threads e o ganho adicional de SIMD.

# 16. Tasks

Tasks são adequadas para trabalho irregular, recursão, descoberta dinâmica de trabalho e DAGs.

```c
#pragma omp task
{
    ...
}
```

É comum combinar:

```c
#pragma omp parallel
{
    #pragma omp single
    {
        // criação de tasks
    }
}
```

### Paralelismo baseado em unidades de trabalho

`parallel for` pressupõe um espaço de iterações conhecido e relativamente estruturado. Tasks permitem descrever **unidades de trabalho que podem ser executadas posteriormente por qualquer thread da equipe**.

Quando uma thread encontra `#pragma omp task`, ela cria uma task. O runtime decide se ela será executada imediatamente ou colocada em uma fila para outra thread.

Por isso, é comum que uma única thread percorra uma estrutura e gere tasks, enquanto toda a equipe participa da execução. A combinação `parallel` + `single` evita que todas as threads gerem o mesmo conjunto de tasks.

Tasks são especialmente úteis para recursão, árvores, algoritmos em blocos, pipelines e cargas irregulares.

In [ ]:
%%writefile omp_task_basic.c
#include <stdio.h>
#include <omp.h>

void trabalho(int id) {
    for (volatile long k=0; k<10000000L; k++);
    printf("task %d -> thread %d\n",
           id, omp_get_thread_num());
}

int main(void) {
    #pragma omp parallel
    {
        #pragma omp single
        {
            for (int i=0; i<8; i++) {
                #pragma omp task firstprivate(i)
                trabalho(i);
            }
            printf("criador terminou de gerar tasks\n");
        }
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_task_basic.c -o omp_task_basic
!OMP_NUM_THREADS=4 ./omp_task_basic

## 16.1 `taskwait`

### Esperando tasks filhas

A criação de uma task é assíncrona em relação à thread que a encontrou. Portanto, depois de criar duas tasks, não podemos assumir que ambas já concluíram.

`taskwait` faz a task atual aguardar a conclusão de suas tasks filhas relevantes antes de continuar.

No exemplo, `a+b` só pode ser calculado depois que as tasks que produzem `a` e `b` terminarem. Sem a sincronização, o `printf` poderia observar os valores antigos.

In [ ]:
%%writefile omp_taskwait.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    int a=0, b=0;

    #pragma omp parallel
    {
        #pragma omp single
        {
            #pragma omp task shared(a)
            a = 10;

            #pragma omp task shared(b)
            b = 20;

            #pragma omp taskwait

            printf("a+b=%d\n", a+b);
        }
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_taskwait.c -o omp_taskwait
!OMP_NUM_THREADS=4 ./omp_taskwait

## 16.2 `taskgroup`

### Esperando um conjunto completo de tasks

`taskgroup` define uma região na qual queremos aguardar as tasks geradas e também seus descendentes relevantes antes de sair da região.

Ele é útil quando uma operação gera uma árvore de tasks e precisamos saber que **todo aquele conjunto de trabalho** terminou.

A diferença para `taskwait` se torna importante em algoritmos recursivos e estruturas de tasks mais profundas.

In [ ]:
%%writefile omp_taskgroup.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        #pragma omp single
        {
            #pragma omp taskgroup
            {
                for (int i=0; i<6; i++) {
                    #pragma omp task firstprivate(i)
                    {
                        for (volatile long k=0; k<5000000L; k++);
                        printf("task %d terminou\n", i);
                    }
                }
            }
            printf("taskgroup terminou\n");
        }
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_taskgroup.c -o omp_taskgroup
!OMP_NUM_THREADS=4 ./omp_taskgroup

### Exercícios 16 — tasks

**16.1.** Compare 100 tasks pequenas com 10 tasks grandes.

**16.2.** Faça cada task ter duração diferente.

**16.3.** Remova `taskwait`. O resultado continua garantido?

**16.4.** Compare `taskwait` e `taskgroup`.

**16.5.** Modele processamento de arquivos de tamanhos diferentes.

**16.6.** Divida uma matriz em blocos e crie uma task por bloco.

**16.7.** Compare workload irregular com `schedule(dynamic)` e tasks.

**16.8.** Explique em que cenário tasks são mais naturais que `parallel for`.

**16.9.** Verifique quais variáveis são `firstprivate` por padrão nas tasks.

# 17. Granularidade de tasks e recursão

Tasks possuem overhead. Um **cutoff** evita gerar tasks excessivamente pequenas.

### Por que tasks muito pequenas são ruins

Criar, registrar e agendar uma task possui custo. Se a própria tarefa executa apenas algumas instruções, o overhead pode ser maior do que o trabalho útil.

Algoritmos recursivos costumam gerar um número exponencial de tasks pequenas. Um *cutoff* limita a criação de tasks: acima de determinado tamanho usamos paralelismo; abaixo dele executamos sequencialmente.

O melhor cutoff não é uma propriedade apenas do algoritmo. Ele depende do processador, número de threads, compilador, runtime e custo do trabalho.

Este tópico introduz um princípio geral de programação paralela: **granularidade adequada é tão importante quanto quantidade de paralelismo**.

In [ ]:
%%writefile omp_task_fib.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

long fib_seq(int n) {
    if (n < 2) return n;
    return fib_seq(n-1) + fib_seq(n-2);
}

long fib_task(int n, int cutoff) {
    if (n < cutoff)
        return fib_seq(n);

    long x, y;

    #pragma omp task shared(x) firstprivate(n,cutoff)
    x = fib_task(n-1, cutoff);

    #pragma omp task shared(y) firstprivate(n,cutoff)
    y = fib_task(n-2, cutoff);

    #pragma omp taskwait
    return x+y;
}

int main(int argc, char **argv) {
    int n = (argc > 1) ? atoi(argv[1]) : 38;
    int cutoff = (argc > 2) ? atoi(argv[2]) : 25;

    long r=0;
    double t0 = omp_get_wtime();

    #pragma omp parallel
    {
        #pragma omp single
        r = fib_task(n, cutoff);
    }

    double t1 = omp_get_wtime();

    printf("fib(%d)=%ld cutoff=%d tempo=%.6f\n",
           n, r, cutoff, t1-t0);
    return 0;
}

In [ ]:
!gcc -O3 -fopenmp omp_task_fib.c -o omp_task_fib
for c in 15 20 25 30 35; do
    OMP_NUM_THREADS=4 ./omp_task_fib 38 $c
done

### Exercícios 17 — granularidade

**17.1.** Faça `cutoff × tempo`.

**17.2.** Conte o número de tasks.

**17.3.** Compare 1, 2, 4 e 8 threads para vários cutoffs.

**17.4.** Implemente Merge Sort com tasks.

**17.5.** Implemente Quicksort com tasks.

**17.6.** Reescreva o cutoff usando a cláusula `if`.

**17.7.** Investigue `final` e `omp_in_final()`.

**17.8.** Explique por que Fibonacci é exemplo didático, mas um algoritmo ruim para Fibonacci real.

**17.9.** Determine experimentalmente um tamanho mínimo de trabalho para compensar a criação de uma task.

# 18. Dependências entre tasks

Tasks podem formar um DAG.

```c
#pragma omp task depend(out:a)
produzir(a);

#pragma omp task depend(in:a) depend(out:b)
transformar(a,b);
```

### Expressando um DAG para o runtime

Em algoritmos mais complexos, algumas tasks podem executar em paralelo e outras dependem de dados produzidos anteriormente.

A cláusula `depend` permite declarar essas relações diretamente. Uma task com `depend(out:a)` produz `a`; uma task com `depend(in:a)` só pode executar quando a produção anterior correspondente estiver concluída.

Com isso, o runtime constrói implicitamente um **grafo acíclico dirigido (DAG)** de trabalho. Tasks independentes podem ser executadas ao mesmo tempo, enquanto as dependências preservam a ordem necessária.

Isso é mais flexível que inserir uma barreira global entre todas as fases, porque apenas as tasks que realmente dependem umas das outras precisam esperar.

In [ ]:
%%writefile omp_task_depend.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    int a=0, b=0, c=0;

    #pragma omp parallel
    {
        #pragma omp single
        {
            #pragma omp task depend(out:a)
            {
                a = 10;
                printf("produziu a\n");
            }

            #pragma omp task depend(in:a) depend(out:b)
            {
                b = 2*a;
                printf("produziu b\n");
            }

            #pragma omp task depend(in:a) depend(out:c)
            {
                c = 3*a;
                printf("produziu c\n");
            }

            #pragma omp task depend(in:b,c)
            {
                printf("resultado=%d\n", b+c);
            }
        }
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_task_depend.c -o omp_task_depend
!OMP_NUM_THREADS=4 ./omp_task_depend

### Exercícios 18 — DAG

**18.1.** Desenhe o DAG.

**18.2.** Identifique tasks simultâneas.

**18.3.** Adicione uma task dependente apenas de `b`.

**18.4.** Modele pipeline carregar → preprocessar → analisar → salvar para vários blocos.

**18.5.** Pesquise fatoração em blocos com tasks.

**18.6.** Modele um stencil temporal em blocos.

**18.7.** Compare dependências específicas com uma barreira global.

**18.8.** Crie um DAG em forma de diamante e meça a concorrência disponível.

# 19. `taskloop`

`taskloop` transforma partes de um laço em tasks.

Parâmetros úteis:

- `grainsize`
- `num_tasks`

### Um laço distribuído como tasks

`taskloop` ocupa uma posição intermediária entre `parallel for` e criação manual de tasks. O programador escreve um laço, e o runtime agrupa suas iterações em tasks.

`grainsize` controla aproximadamente quanto trabalho cada task contém. `num_tasks` permite orientar o runtime pela quantidade desejada de tasks.

Isso é particularmente útil quando queremos combinar a sintaxe de um laço com o agendamento dinâmico e a composição de tasks.

A pergunta experimental é semelhante à escolha de chunk em `schedule`: granularidade muito fina aumenta overhead; granularidade muito grossa pode reduzir balanceamento.

In [ ]:
%%writefile omp_taskloop.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(void) {
    const long N = 5000000L;
    double *a = malloc(N*sizeof(double));
    double soma = 0.0;

    #pragma omp parallel
    {
        #pragma omp single
        {
            #pragma omp taskloop grainsize(100000) reduction(+:soma)
            for (long i=0; i<N; i++) {
                a[i] = 1.0/(i+1);
                soma += a[i];
            }
        }
    }

    printf("soma=%.12f\n", soma);
    free(a);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_taskloop.c -o omp_taskloop
!OMP_NUM_THREADS=4 ./omp_taskloop

### Exercícios 19 — `taskloop`

**19.1.** Varie `grainsize`.

**19.2.** Compare `grainsize` e `num_tasks`.

**19.3.** Compare `taskloop` e `parallel for schedule(dynamic)`.

**19.4.** Use carga irregular.

**19.5.** Reimplemente contagem de primos.

**19.6.** Explique quando `taskloop` simplifica o código.

**19.7.** Diferencie chunk de `schedule` e granularidade de tasks.

**19.8.** Investigue a sincronização implícita associada ao `taskloop`.

# 20. Locks

OpenMP também oferece locks explícitos:

- `omp_init_lock`
- `omp_set_lock`
- `omp_unset_lock`
- `omp_destroy_lock`
- `omp_test_lock`

Locks oferecem controle fino, mas normalmente devemos preferir abstrações de mais alto nível quando elas expressam corretamente o problema.

### Sincronização explícita

Locks permitem que o programa controle diretamente a aquisição e liberação de exclusão mútua. Uma thread chama `omp_set_lock()` e pode entrar na região protegida quando obtém o lock; ao terminar, chama `omp_unset_lock()`.

Esse mecanismo é poderoso porque diferentes recursos podem possuir locks diferentes. Contudo, a flexibilidade aumenta o risco de erros, incluindo esquecer de liberar o lock ou criar deadlocks.

Um deadlock pode ocorrer quando duas threads seguram recursos diferentes e cada uma espera pelo recurso da outra.

Prefira `reduction`, `atomic` e `critical` quando eles expressarem naturalmente o problema. Use locks quando o algoritmo realmente exigir controle explícito sobre recursos independentes.

In [ ]:
%%writefile omp_lock.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    omp_lock_t lock;
    omp_init_lock(&lock);

    long contador = 0;

    #pragma omp parallel for
    for (long i=0; i<1000000L; i++) {
        omp_set_lock(&lock);
        contador++;
        omp_unset_lock(&lock);
    }

    omp_destroy_lock(&lock);

    printf("contador=%ld\n", contador);
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_lock.c -o omp_lock
!OMP_NUM_THREADS=4 ./omp_lock

### Exercícios 20 — locks

**20.1.** Compare lock, `critical`, `atomic` e `reduction`.

**20.2.** Crie dois recursos protegidos por locks diferentes.

**20.3.** Construa um exemplo de deadlock com dois locks em ordem inversa. Não deixe o programa executando indefinidamente.

**20.4.** Use `omp_test_lock`.

**20.5.** Modele várias filas independentes, cada uma com lock próprio.

**20.6.** Explique por que um lock global pode destruir escalabilidade.

**20.7.** Explique o princípio de sempre adquirir múltiplos locks em uma ordem consistente.

# 21. Afinidade de threads

Variáveis úteis:

```text
OMP_DISPLAY_ENV=TRUE
OMP_PROC_BIND=true
OMP_PROC_BIND=close
OMP_PROC_BIND=spread
OMP_PLACES=cores
OMP_PLACES=threads
```

Afinidade pode alterar:

- localidade de cache;
- migração de threads;
- compartilhamento de recursos;
- comportamento NUMA.

### Onde uma thread executa importa

O sistema operacional pode mover threads entre CPUs. Em programas paralelos de alto desempenho, essas migrações podem prejudicar reutilização de cache e tornar resultados menos estáveis.

Afinidade (*thread affinity*) permite controlar ou orientar onde as threads OpenMP são posicionadas.

`OMP_PLACES=cores` descreve os locais como cores físicos. `OMP_PROC_BIND=close` tende a manter threads próximas, enquanto `spread` tende a espalhá-las pelos locais disponíveis.

Em sistemas com SMT, múltiplos sockets ou NUMA, a política de colocação pode alterar significativamente o desempenho. Não existe uma configuração universalmente melhor: aplicações que compartilham dados e aplicações que competem por recursos podem preferir posicionamentos diferentes.

In [ ]:
%%writefile omp_affinity.c
#define _GNU_SOURCE
#include <stdio.h>
#include <sched.h>
#include <omp.h>

int main(void) {
    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        int cpu = sched_getcpu();

        #pragma omp critical
        printf("thread=%d cpu=%d place=%d\n",
               tid, cpu, omp_get_place_num());
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_affinity.c -o omp_affinity

echo "---- close"
OMP_NUM_THREADS=4 OMP_PROC_BIND=close OMP_PLACES=cores ./omp_affinity

echo "---- spread"
OMP_NUM_THREADS=4 OMP_PROC_BIND=spread OMP_PLACES=cores ./omp_affinity

### Exercícios 21 — afinidade

**21.1.** Compare `close` e `spread`.

**21.2.** Compare `OMP_PLACES=cores` e `OMP_PLACES=threads`.

**21.3.** Use `OMP_DISPLAY_ENV=VERBOSE`.

**21.4.** Repita SAXPY com diferentes políticas.

**21.5.** Repita contagem de primos.

**21.6.** Discuta limitações de experimentos de afinidade no Colab.

**21.7.** Explique o que pode ocorrer em uma máquina multi-socket/NUMA.

**21.8.** Investigue `proc_bind` como cláusula em uma região `parallel`.

# 22. False sharing e localidade

Caches trabalham em linhas de cache.

Se threads atualizam variáveis diferentes que ficam na mesma linha, pode ocorrer **false sharing**.

O programa continua correto, mas o desempenho pode cair muito.

### Quando dados diferentes ainda entram em conflito

Caches transferem dados em blocos chamados **linhas de cache**, tipicamente dezenas de bytes. Duas variáveis distintas podem ocupar a mesma linha.

Se duas threads atualizam continuamente variáveis diferentes da mesma linha, o protocolo de coerência pode transferir essa linha repetidamente entre os caches. Esse fenômeno é chamado *false sharing* porque as threads não compartilham logicamente a variável, mas compartilham fisicamente a linha.

Padding ou reorganização dos dados pode separar as variáveis em linhas distintas.

False sharing é diferente de race condition: a execução continua semanticamente correta, mas pode ficar muito mais lenta. Essa distinção é fundamental em otimização de códigos paralelos.

In [ ]:
%%writefile omp_false_sharing.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

typedef struct {
    volatile long value;
} compact_t;

typedef struct {
    volatile long value;
    char padding[64];
} padded_t;

int main(void) {
    const long ITER = 20000000L;
    int nt = omp_get_max_threads();

    compact_t *a = calloc(nt, sizeof(compact_t));
    padded_t  *b = calloc(nt, sizeof(padded_t));

    double t0 = omp_get_wtime();

    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        for (long i=0; i<ITER; i++)
            a[tid].value++;
    }

    double t1 = omp_get_wtime();

    #pragma omp parallel
    {
        int tid = omp_get_thread_num();
        for (long i=0; i<ITER; i++)
            b[tid].value++;
    }

    double t2 = omp_get_wtime();

    printf("compacto=%.6f padded=%.6f\n", t1-t0, t2-t1);

    free(a); free(b);
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_false_sharing.c -o omp_false_sharing
!OMP_NUM_THREADS=4 OMP_PROC_BIND=close OMP_PLACES=cores ./omp_false_sharing

### Exercícios 22 — cache e localidade

**22.1.** Teste 1, 2, 4 e 8 threads.

**22.2.** Varie o padding.

**22.3.** Explique o uso de `volatile` neste microbenchmark.

**22.4.** Faça redução manual com slots por thread adjacentes e separados.

**22.5.** Compare percorrer matriz por linhas e por colunas.

**22.6.** Compare `schedule(static)` e `schedule(static,1)` em vetor grande.

**22.7.** Investigue first-touch em uma máquina NUMA.

**22.8.** Diferencie race condition e false sharing.

**22.9.** Meça largura de banda efetiva para diferentes números de threads.

# 23. Paralelismo aninhado

Uma região paralela pode conter outra região paralela.

Use com cuidado para evitar oversubscription.

OpenMP moderno permite controlar níveis ativos com:

```c
omp_set_max_active_levels(...)
```

### Equipes dentro de equipes

Uma thread que já participa de uma região paralela pode encontrar outra região `parallel`. Se níveis adicionais estiverem ativos, uma nova equipe pode ser criada.

Isso pode ser útil em algoritmos hierárquicos, mas também pode multiplicar rapidamente o número de threads. Por exemplo, quatro threads externas criando quatro threads internas podem produzir até dezesseis contextos concorrentes.

Se o hardware possui apenas quatro ou oito CPUs disponíveis, esse excesso causa *oversubscription*: mais threads prontas do que recursos de execução, aumentando trocas de contexto e disputa.

Em laços regulares aninhados, `collapse` costuma ser uma solução mais simples. Paralelismo aninhado é mais apropriado quando existem níveis realmente distintos de paralelismo.

In [ ]:
%%writefile omp_nested.c
#include <stdio.h>
#include <omp.h>

int main(void) {
    omp_set_max_active_levels(2);

    #pragma omp parallel num_threads(2)
    {
        int outer = omp_get_thread_num();

        #pragma omp parallel num_threads(2)
        {
            printf("outer=%d inner=%d level=%d active=%d\n",
                   outer,
                   omp_get_thread_num(),
                   omp_get_level(),
                   omp_get_active_level());
        }
    }
    return 0;
}

In [ ]:
!gcc -O2 -fopenmp omp_nested.c -o omp_nested
!./omp_nested

### Exercícios 23 — nested

**23.1.** Modifique o número de threads dos dois níveis.

**23.2.** Use `omp_get_level()` e `omp_get_active_level()`.

**23.3.** Calcule o total potencial de threads.

**23.4.** Compare `collapse(2)` e paralelismo aninhado.

**23.5.** Explique por que `collapse` tende a ser mais simples em laços regulares.

**23.6.** Analise o caso em que uma biblioteca chamada pelo seu código também usa threads.

**23.7.** Explique oversubscription e seus efeitos.

# 24. Estudo de caso — integração numérica de $\pi$

Aproximação:

$$
\pi = \int_0^1 \frac{4}{1+x^2}\,dx
$$

Com $N$ retângulos:

$$
\pi \approx h \sum_{i=0}^{N-1}\frac{4}{1+x_i^2}
$$

onde:

$$
h = \frac{1}{N}
$$

e:

$$
x_i = \left(i+\frac{1}{2}\right)h
$$

### O que este estudo de caso combina

A aproximação numérica de $\pi$ transforma uma integral em uma soma de muitas contribuições independentes. Cada iteração calcula um valor de $x_i$ e sua contribuição para a integral.

Isso torna o problema ideal para estudar `reduction`: o cálculo local é independente, mas todas as contribuições precisam ser combinadas.

O caso também permite comparar diferentes mecanismos de sincronização e observar como uma solução correta pode ter desempenho muito diferente dependendo da estratégia usada.

Além disso, é uma oportunidade para discutir precisão de ponto flutuante: paralelizar a soma muda a ordem das operações e pode alterar levemente os últimos dígitos.

In [ ]:
%%writefile omp_pi.c
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <omp.h>

int main(int argc, char **argv) {
    long N = (argc > 1) ? atol(argv[1]) : 20000000L;
    double h = 1.0/N;
    double soma = 0.0;

    double t0 = omp_get_wtime();

    #pragma omp parallel for reduction(+:soma)
    for (long i=0; i<N; i++) {
        double x = (i+0.5)*h;
        soma += 4.0/(1.0+x*x);
    }

    double pi = h*soma;
    double t1 = omp_get_wtime();

    printf("threads=%d N=%ld pi=%.15f erro=%.3e tempo=%.6f\n",
           omp_get_max_threads(), N, pi, fabs(pi-M_PI), t1-t0);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_pi.c -lm -o omp_pi
for t in 1 2 4 8; do
    OMP_NUM_THREADS=$t ./omp_pi 10000000
done

### Exercícios 24 — $\pi$

Implemente e compare:

**24.1.** sequencial;

**24.2.** `critical`;

**24.3.** `atomic`;

**24.4.** redução manual;

**24.5.** `reduction`;

**24.6.** `parallel for simd reduction`;

**24.7.** `float` versus `double`;

**24.8.** múltiplos valores de $N$;

**24.9.** speedup e eficiência;

**24.10.** erro numérico;

**24.11.** ordem da soma em ponto flutuante;

**24.12.** interpretação com Lei de Amdahl.

# 25. Estudo de caso — histograma

Para dados entre 0 e 255:

```c
hist[data[i]]++;
```

é uma atualização concorrente em estado compartilhado.

Este caso permite comparar sincronização e privatização.

### Paralelismo com contenção dependente dos dados

No histograma, cada elemento da entrada determina qual contador será incrementado. As leituras da entrada são independentes, mas os contadores são compartilhados.

Se muitos elementos mapeiam para o mesmo bin, várias threads tentam atualizar a mesma posição, aumentando contenção. Por isso, o desempenho não depende apenas do tamanho da entrada, mas também da **distribuição dos valores**.

Uma otimização clássica é privatizar o histograma: cada thread mantém seus próprios bins e, no final, os histogramas locais são combinados. Isso troca muitas sincronizações finas por uma fase de redução mais estruturada.

O exemplo conecta sincronização, localidade, false sharing e características dos dados.

In [ ]:
%%writefile omp_histogram.c
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <omp.h>

#define BINS 256

int main(int argc, char **argv) {
    long N = (argc > 1) ? atol(argv[1]) : 10000000L;

    uint8_t *data = malloc(N*sizeof(uint8_t));
    long hist[BINS] = {0};

    for (long i=0; i<N; i++)
        data[i] = (uint8_t)(i % BINS);

    double t0 = omp_get_wtime();

    #pragma omp parallel for
    for (long i=0; i<N; i++) {
        #pragma omp atomic
        hist[data[i]]++;
    }

    double t1 = omp_get_wtime();

    long total=0;
    for (int b=0; b<BINS; b++)
        total += hist[b];

    printf("total=%ld tempo=%.6f\n", total, t1-t0);

    free(data);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_histogram.c -o omp_histogram
!OMP_NUM_THREADS=4 ./omp_histogram 5000000

### Exercícios 25 — histograma

**25.1.** Versão sequencial.

**25.2.** Versão com `critical`.

**25.3.** Versão com `atomic`.

**25.4.** Histograma privado por thread + combinação final.

**25.5.** Investigue false sharing nos histogramas privados.

**25.6.** Compare dados uniformes, concentrados em um bin e concentrados em quatro bins.

**25.7.** Explique o impacto da distribuição dos dados.

**25.8.** Teste diferentes números de bins.

**25.9.** Gere `threads × throughput`.

**25.10.** Aplique a uma imagem PGM.

# 26. Estudo de caso — stencil 2D

Stencil de cinco pontos:

$$
B_{i,j} =
\frac{
A_{i,j}+
A_{i-1,j}+
A_{i+1,j}+
A_{i,j-1}+
A_{i,j+1}
}{5}
$$

Esse padrão aparece em difusão, métodos numéricos, imagens e PDEs.

### Um padrão típico de HPC

Em um stencil, cada célula nova é calculada a partir de uma pequena vizinhança da grade anterior. Dentro de um mesmo passo temporal, as células de saída normalmente podem ser calculadas em paralelo.

Quando existem vários passos temporais, surge uma dependência entre fases: o passo $t+1$ precisa dos resultados do passo $t$. Portanto, uma sincronização entre passos é necessária.

Stencils movimentam muitos dados em relação à quantidade de operações aritméticas e frequentemente são limitados por memória. Eles também são excelentes para estudar localidade, ordem dos laços, SIMD e blocking.

É um exemplo próximo de aplicações científicas reais como difusão, transferência de calor e solução numérica de PDEs.

In [ ]:
%%writefile omp_stencil.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 1500;
    size_t total = (size_t)N*N;

    double *A = malloc(total*sizeof(double));
    double *B = malloc(total*sizeof(double));

    #pragma omp parallel for
    for (size_t k=0; k<total; k++)
        A[k] = 1.0;

    double t0 = omp_get_wtime();

    #pragma omp parallel for schedule(static)
    for (int i=1; i<N-1; i++) {
        for (int j=1; j<N-1; j++) {
            size_t k = (size_t)i*N+j;
            B[k] = (
                A[k] +
                A[k-N] +
                A[k+N] +
                A[k-1] +
                A[k+1]
            ) / 5.0;
        }
    }

    double t1 = omp_get_wtime();

    printf("N=%d threads=%d tempo=%.6f B=%f\n",
           N, omp_get_max_threads(), t1-t0,
           B[(size_t)(N/2)*N+(N/2)]);

    free(A); free(B);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_stencil.c -o omp_stencil
!OMP_NUM_THREADS=4 ./omp_stencil 1000

### Exercícios 26 — stencil

**26.1.** Faça múltiplos passos temporais trocando `A` e `B`.

**26.2.** Identifique a barreira necessária entre passos.

**26.3.** Compare schedules.

**26.4.** Compare laço de linhas com `collapse(2)`.

**26.5.** Adicione `simd` no laço interno.

**26.6.** Use `parallel for simd`.

**26.7.** Calcule células atualizadas por segundo.

**26.8.** Estime bytes por atualização.

**26.9.** Classifique como memory-bound ou compute-bound.

**26.10.** Implemente difusão de calor.

**26.11.** Implemente filtro de média 3×3.

**26.12.** Faça blocking/tiling.

# 27. Estudo de caso — Mandelbrot

Para cada ponto complexo $c$:

$$
z_{n+1} = z_n^2 + c
$$

Cada pixel pode exigir um número diferente de iterações. Portanto, a carga é irregular.

### Paralelismo abundante, mas irregular

Cada pixel do conjunto de Mandelbrot pode ser calculado independentemente, então existe muito paralelismo.

O problema é que diferentes pixels exigem números muito diferentes de iterações antes de escapar ou atingir `max_iter`. Regiões próximas à fronteira do conjunto tendem a demandar mais trabalho.

Assim, uma divisão estática perfeitamente uniforme em número de pixels pode ser altamente desigual em tempo de computação.

Mandelbrot é um excelente laboratório para demonstrar que **balancear a quantidade de itens não é necessariamente balancear a quantidade de trabalho** e para comparar `static`, `dynamic`, `guided` e diferentes chunks.

In [ ]:
%%writefile omp_mandelbrot.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int mandel(double cr, double ci, int max_iter) {
    double zr=0.0, zi=0.0;

    for (int it=0; it<max_iter; it++) {
        double zr2 = zr*zr - zi*zi + cr;
        double zi2 = 2.0*zr*zi + ci;
        zr = zr2;
        zi = zi2;

        if (zr*zr + zi*zi > 4.0)
            return it;
    }
    return max_iter;
}

int main(int argc, char **argv) {
    int W = (argc > 1) ? atoi(argv[1]) : 800;
    int H = W;
    int max_iter = 1000;

    int *img = malloc((size_t)W*H*sizeof(int));

    double t0 = omp_get_wtime();

    #pragma omp parallel for schedule(runtime)
    for (int y=0; y<H; y++) {
        double ci = -1.2 + 2.4*y/(H-1.0);

        for (int x=0; x<W; x++) {
            double cr = -2.0 + 3.0*x/(W-1.0);
            img[(size_t)y*W+x] = mandel(cr, ci, max_iter);
        }
    }

    double t1 = omp_get_wtime();

    long checksum=0;
    for (size_t k=0; k<(size_t)W*H; k++)
        checksum += img[k];

    printf("W=%d tempo=%.6f checksum=%ld\n",
           W, t1-t0, checksum);

    free(img);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp omp_mandelbrot.c -o omp_mandelbrot
for sched in static dynamic,1 dynamic,4 guided; do
    echo "---- $sched"
    OMP_NUM_THREADS=4 OMP_SCHEDULE=$sched ./omp_mandelbrot 500
done

### Exercícios 27 — Mandelbrot

**27.1.** Compare `static`, `dynamic` e `guided`.

**27.2.** Teste diferentes chunks.

**27.3.** Registre trabalho total por thread.

**27.4.** Calcule desbalanceamento.

**27.5.** Relacione balanceamento e tempo.

**27.6.** Teste `collapse(2)`.

**27.7.** Investigue SIMD.

**27.8.** Gere uma imagem PGM.

**27.9.** Produza uma imagem 4K e meça escalabilidade.

**27.10.** Explique por que `dynamic,1` pode balancear bem sem ser a configuração mais rápida.

# 28. Estudo de caso — multiplicação de matrizes

Para matrizes quadradas:

$$
C_{i,j} = \sum_{k=0}^{N-1} A_{i,k}B_{k,j}
$$

Esse caso permite estudar threads, SIMD, ordem dos laços, blocking e intensidade aritmética.

### Computação, memória e hierarquia de cache

Multiplicação de matrizes possui muito paralelismo, mas seu desempenho depende fortemente de como os dados são acessados.

A ordem dos laços altera o padrão de memória. Como C armazena matrizes por linhas (*row-major*), acessar elementos contíguos é normalmente mais eficiente que percorrer uma coluna com grandes saltos.

Blocking divide as matrizes em blocos menores para aumentar reutilização de dados nos caches. SIMD pode explorar o laço interno e `parallel for` pode distribuir linhas ou blocos entre threads.

Este exemplo mostra que paralelização não substitui otimização sequencial: uma versão com melhor localidade em uma thread pode ser mais eficiente e também escalar melhor.

In [ ]:
%%writefile omp_matmul.c
#include <stdio.h>
#include <stdlib.h>
#include <omp.h>

int main(int argc, char **argv) {
    int N = (argc > 1) ? atoi(argv[1]) : 384;
    size_t total = (size_t)N*N;

    double *A = aligned_alloc(64, total*sizeof(double));
    double *B = aligned_alloc(64, total*sizeof(double));
    double *C = aligned_alloc(64, total*sizeof(double));

    #pragma omp parallel for
    for (size_t k=0; k<total; k++) {
        A[k] = 1.0;
        B[k] = 1.0;
        C[k] = 0.0;
    }

    double t0 = omp_get_wtime();

    #pragma omp parallel for schedule(static)
    for (int i=0; i<N; i++) {
        for (int k=0; k<N; k++) {
            double aik = A[(size_t)i*N+k];

            #pragma omp simd
            for (int j=0; j<N; j++)
                C[(size_t)i*N+j] += aik*B[(size_t)k*N+j];
        }
    }

    double t1 = omp_get_wtime();

    double secs = t1-t0;
    double gflops = (2.0*N*N*N)/(secs*1e9);

    printf("N=%d threads=%d tempo=%.6f GFLOP/s=%.3f C00=%f\n",
           N, omp_get_max_threads(), secs, gflops, C[0]);

    free(A); free(B); free(C);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -fopenmp \
    -fopt-info-vec-optimized omp_matmul.c -o omp_matmul
for t in 1 2 4 8; do
    OMP_NUM_THREADS=$t ./omp_matmul 256
done

### Exercícios 28 — matrizes

**28.1.** Implemente ordem `i-j-k`.

**28.2.** Implemente `i-k-j`.

**28.3.** Explique a localidade.

**28.4.** Paralelize somente o laço externo.

**28.5.** Adicione `simd`.

**28.6.** Compare `parallel for` e `parallel for + simd`.

**28.7.** Implemente blocking com 16, 32, 64 e 128.

**28.8.** Gere `block size × GFLOP/s`.

**28.9.** Compare com BLAS.

**28.10.** Explique por que BLAS otimizada tende a ser mais rápida.

**28.11.** Faça strong scaling.

**28.12.** Faça weak scaling.

**28.13.** Compare `float` e `double`.

**28.14.** Investigue transposição prévia de `B`.

# 29. Automatizando experimentos

Não copie tempos manualmente.

O Python pode:

1. executar o programa;
2. controlar `OMP_NUM_THREADS`;
3. variar `OMP_SCHEDULE`;
4. repetir;
5. calcular estatísticas;
6. gerar gráficos;
7. exportar CSV.

### Por que automatizar

Experimentos paralelos possuem muitas dimensões: tamanho da entrada, número de threads, schedule, chunk, afinidade e repetições.

Copiar resultados manualmente favorece erros e torna difícil reproduzir o estudo. Automatizar com Python permite executar todas as configurações de forma uniforme e guardar os dados brutos.

O script usa variáveis de ambiente para alterar a configuração OpenMP sem recompilar e extrai o tempo impresso pelo executável.

Uma boa automação deve guardar não apenas médias, mas também a configuração completa do experimento. Isso transforma uma demonstração informal em um experimento reproduzível.

In [ ]:
def executar_openmp(comando, threads, repeticoes=5, env_extra=None):
    resultados = []

    env_base = dict(os.environ)
    env_base["OMP_NUM_THREADS"] = str(threads)

    if env_extra:
        env_base.update(env_extra)

    for _ in range(repeticoes):
        proc = subprocess.run(
            comando,
            env=env_base,
            capture_output=True,
            text=True,
            check=True
        )

        m = re.search(r"tempo=([0-9.]+)", proc.stdout)
        if not m:
            raise ValueError(
                "Não encontrei 'tempo=' na saída:\n" + proc.stdout
            )

        resultados.append(float(m.group(1)))

    return resultados

In [ ]:
linhas = []

# Requer omp_pi compilado.
for t in [1, 2, 4, 8]:
    tempos = executar_openmp(
        ["./omp_pi", "5000000"],
        threads=t,
        repeticoes=5
    )

    r = resumo(tempos)
    linhas.append({
        "threads": t,
        "tempo_medio": r["media"],
        "desvio": r["desvio"]
    })

df = pd.DataFrame(linhas)
df["speedup"] = df.loc[0, "tempo_medio"] / df["tempo_medio"]
df["eficiencia"] = df["speedup"] / df["threads"]
df

In [ ]:
plt.errorbar(
    df["threads"],
    df["tempo_medio"],
    yerr=df["desvio"],
    marker="o"
)
plt.xlabel("Threads")
plt.ylabel("Tempo (s)")
plt.title("Tempo × threads")
plt.grid(True, alpha=0.3)
plt.show()

plt.plot(
    df["threads"],
    df["speedup"],
    marker="o",
    label="medido"
)
plt.plot(
    df["threads"],
    df["threads"],
    linestyle="--",
    label="ideal"
)
plt.xlabel("Threads")
plt.ylabel("Speedup")
plt.title("Speedup × threads")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Exercícios 29 — experimentação

**29.1.** Adapte o script para testar `OMP_SCHEDULE`.

**29.2.** Gere média, mediana, mínimo, máximo e desvio-padrão.

**29.3.** Adicione intervalo de confiança de 95%.

**29.4.** Exporte resultados para CSV.

**29.5.** Registre CPU, compilador, flags, data, threads, `OMP_PROC_BIND` e `OMP_PLACES`.

**29.6.** Crie gráfico de eficiência.

**29.7.** Crie heatmap `threads × chunk → tempo`.

**29.8.** Compare três aplicações usando exatamente o mesmo protocolo.

**29.9.** Separe tempo de inicialização do tempo da região paralela.

**29.10.** Crie uma função que detecte automaticamente os números de threads razoáveis para a máquina.

# 30. Protocolo experimental da disciplina

## 30.1 Corretude

Antes de medir:

- compare com implementação sequencial;
- valide todos os elementos ou uma métrica de erro;
- use tolerância apropriada para ponto flutuante.

## 30.2 Compilação

Padrão para desempenho:

```text
-O3 -march=native -fopenmp
```

Para estudar vetorização:

```text
-fopt-info-vec-optimized
-fopt-info-vec-missed
```

## 30.3 Aquecimento

Evite interpretar somente a primeira execução.

## 30.4 Repetições

Use várias repetições e reporte dispersão.

## 30.5 Região medida

Diferencie:

- alocação;
- inicialização;
- I/O;
- computação;
- pós-processamento.

## 30.6 Escalabilidade

Quando possível:

- 1;
- 2;
- 4;
- 8;
- demais valores compatíveis com a máquina.

## 30.7 Conclusão

Não escreva apenas:

> “OpenMP foi mais rápido.”

Informe:

- quanto;
- em qual configuração;
- para qual tamanho;
- com qual variabilidade;
- qual o gargalo;
- por que o comportamento ocorreu.

### Como interpretar desempenho de maneira científica

Um protocolo experimental define regras comuns para que resultados obtidos em exercícios diferentes sejam comparáveis.

A sequência recomendada é:

**corretude → baseline → aquecimento → repetições → estatística → speedup → análise do gargalo.**

Flags de compilação precisam ser mantidas constantes entre versões comparadas. A região medida também precisa ser explicitada: incluir alocação ou I/O em uma versão e excluí-los em outra torna a comparação inválida.

Em escalabilidade forte (*strong scaling*), o tamanho total do problema permanece fixo enquanto aumentamos recursos. Em escalabilidade fraca (*weak scaling*), aumentamos o problema junto com os recursos para manter aproximadamente constante o trabalho por thread.

O objetivo final não é apenas produzir um número de speedup, mas explicar **por que** a aplicação se comporta daquela forma.

# 31. Banco de exercícios — vetores e estatística

Para cada aplicação: versão sequencial, versão OpenMP, validação e speedup.

### Vetores

1. soma;
2. SAXPY;
3. produto escalar;
4. norma L1;
5. norma L2;
6. distância Euclidiana;
7. normalização;
8. threshold;
9. clamp;
10. transformação afim;
11. prefixo simples para discutir dependências;
12. busca de valor;
13. contagem por condição.

### Estatística

14. soma;
15. média;
16. mínimo;
17. máximo;
18. variância;
19. desvio-padrão;
20. histograma;
21. média ponderada;
22. covariância de dois vetores;
23. erro quadrático médio;
24. correlação simplificada.

### Como usar este banco

Estes exercícios consolidam os construtos básicos em problemas deliberadamente simples. O objetivo é que a complexidade do domínio da aplicação não esconda o comportamento do OpenMP.

Para cada problema, comece por uma versão sequencial correta e só depois introduza paralelismo. Ao terminar, o aluno deve conseguir justificar qual construto OpenMP foi escolhido e qual recurso de hardware provavelmente limita o desempenho.

# 32. Banco de exercícios — imagens, matrizes e cargas irregulares

### Imagens

1. negativo;
2. threshold;
3. brilho;
4. contraste;
5. histograma;
6. blur;
7. Sobel;
8. convolução 3×3;
9. detecção simples de bordas;
10. diferença entre duas imagens.

### Matrizes

11. inicialização;
12. soma;
13. transposição;
14. matriz-vetor;
15. matriz-matriz;
16. stencil;
17. soma por linhas;
18. soma por colunas;
19. normalização por linha.

### Irregulares

20. primos;
21. Mandelbrot;
22. busca em listas de tamanhos distintos;
23. arquivos de tamanhos distintos;
24. árvores;
25. grafos simples;
26. simulação de trabalhos com durações diferentes.

Para cada problema, classifique o gargalo predominante:

- computação;
- memória;
- sincronização;
- desbalanceamento;
- overhead.

### O que muda nestas aplicações

Aqui os problemas começam a apresentar padrões de acesso à memória e balanceamento mais realistas. Imagens e matrizes permitem estudar localidade e SIMD; cargas irregulares tornam `schedule` e tasks mais relevantes.

A intenção não é resolver todos os exercícios, mas usar aplicações diferentes para perceber que a melhor estratégia OpenMP depende da estrutura do algoritmo e dos dados.

# 33. Banco de exercícios — SIMD, tasks e localidade

### SIMD

1. SAXPY;
2. produto escalar;
3. redução;
4. normalização;
5. filtro 1D;
6. função polinomial;
7. comparação `float` × `double`;
8. branch em vetorização.

### Tasks

9. Merge Sort;
10. Quicksort;
11. árvore;
12. travessia de diretórios;
13. pipeline;
14. DAG em diamante;
15. processamento de blocos.

### Localidade

16. ordem de laços em matrizes;
17. blocking;
18. false sharing;
19. first-touch;
20. afinidade;
21. `schedule(static,1)` versus blocos contíguos.

### Combinação

22. aplicação com threads + SIMD;
23. aplicação com tasks + dependências;
24. aplicação com schedule + reduction;
25. aplicação com afinidade + análise de cache.

### Objetivo desta etapa

Estes exercícios combinam mecanismos que atuam em níveis diferentes. SIMD explora paralelismo vetorial dentro de um core; tasks descrevem trabalho dinâmico; localidade determina quão eficientemente os dados chegam às unidades de execução.

O aluno deve começar a pensar em OpenMP não como uma coleção isolada de pragmas, mas como um conjunto de ferramentas que podem ser combinadas de acordo com o gargalo observado.

# 34. Exercício integrador — do sequencial ao OpenMP otimizado

Escolha:

- Mandelbrot;
- stencil;
- histograma;
- matriz-matriz;
- processamento de imagem.

## Etapa A — baseline

Implemente e valide a versão sequencial.

## Etapa B — decomposição

Identifique:

- trabalho;
- dados;
- dependências;
- regiões paralelizáveis.

## Etapa C — OpenMP básico

Use o construto mais simples correto.

## Etapa D — schedule

Compare políticas quando aplicável.

## Etapa E — sincronização

Escolha adequadamente entre `atomic`, `critical` e `reduction`.

## Etapa F — SIMD

Verifique a vetorização e o ganho real.

## Etapa G — localidade

Analise acesso à memória e false sharing.

## Etapa H — escalabilidade

Produza tempo, speedup e eficiência.

## Etapa I — análise

Responda:

1. Onde está o gargalo?
2. Qual otimização mais ajudou?
3. Qual otimização não ajudou?
4. Por quê?
5. Até quantas threads vale a pena escalar?

### Metodologia do exercício integrador

Este exercício reproduz um fluxo real de otimização. Não se começa adicionando todas as diretivas disponíveis. Primeiro estabelecemos um baseline correto, depois identificamos paralelismo, medimos, localizamos o gargalo e somente então aplicamos otimizações adicionais.

Cada etapa deve manter a corretude da etapa anterior. Se uma otimização produz speedup, o aluno deve conseguir explicar o mecanismo responsável pelo ganho.

# 35. Exercícios de depuração

Os fragmentos abaixo devem ser **diagnosticados e corrigidos**.

## 35.1 Race

```c
double soma = 0;
#pragma omp parallel for
for (int i=0; i<N; i++)
    soma += a[i];
```

## 35.2 Dependência

```c
#pragma omp parallel for
for (int i=1; i<N; i++)
    a[i] = a[i-1] + 1;
```

## 35.3 Variável temporária compartilhada

```c
int tmp;
#pragma omp parallel for
for (int i=0; i<N; i++) {
    tmp = f(i);
    a[i] = tmp;
}
```

## 35.4 `nowait` incorreto

```c
#pragma omp for nowait
for (...)
    produzir(a);

#pragma omp for
for (...)
    consumir(a);
```

## 35.5 Task sem dependência

Uma task consome uma estrutura antes da task produtora terminar.

Para cada caso:

1. explique o erro;
2. faça versão correta;
3. apresente soluções alternativas;
4. compare custo quando aplicável.

### Por que depurar código paralelo é diferente

Erros paralelos podem depender de timing e desaparecer quando adicionamos mensagens de depuração. Por isso, a análise deve começar pelas garantias do modelo de memória e pelas dependências do algoritmo.

Nestes exercícios, o objetivo não é apenas encontrar “qual pragma falta”, mas identificar exatamente qual relação de ordenação ou privacidade foi violada.

# 36. Mini-projetos OpenMP

## Projeto A — processamento de imagem

Pipeline:

1. leitura;
2. escala de cinza;
3. filtro;
4. histograma;
5. threshold;
6. escrita.

---

## Projeto B — difusão de calor

Grade 2D com múltiplos passos.

Avalie:

- `parallel for`;
- `collapse`;
- SIMD;
- afinidade;
- escalabilidade.

---

## Projeto C — Mandelbrot

Avalie:

- `schedule`;
- chunk;
- balanceamento;
- SIMD;
- escalabilidade.

---

## Projeto D — álgebra linear

Implemente:

- matriz-vetor;
- matriz-matriz;
- blocking;
- SIMD.

Compare com biblioteca otimizada.

---

## Projeto E — conjunto irregular de trabalhos

Compare:

- `dynamic`;
- `guided`;
- tasks;
- `taskloop`.

---

## Projeto F — pipeline orientado a tasks

Modele estágios com dependências explícitas e múltiplos blocos em voo simultaneamente.

### Objetivo dos mini-projetos

Os mini-projetos exigem combinar vários tópicos do notebook em uma aplicação maior. O foco deve ser tanto a implementação quanto a análise experimental.

Uma solução mais complexa só é considerada melhor se continuar correta e apresentar evidência mensurável de benefício. Diretivas adicionais sem ganho ou necessidade algorítmica não constituem otimização.

# 37. Entrega experimental sugerida

Para cada mini-projeto:

## Implementações

- sequencial;
- OpenMP básica;
- OpenMP otimizada.

## Experimentos

Pelo menos:

- 3 tamanhos de entrada;
- 4 configurações de threads, se a máquina permitir;
- 5 repetições por configuração.

## Figuras

1. tempo × threads;
2. speedup × threads;
3. eficiência × threads;
4. figura específica da aplicação.

## Discussão

Explique:

- limite de escalabilidade;
- overhead;
- balanceamento;
- sincronização;
- localidade;
- vetorização;
- impacto do tamanho da entrada.

### O que se espera da entrega

A entrega deve permitir que outra pessoa reproduza o experimento e entenda as decisões tomadas. Código, configuração, entrada e metodologia de medição fazem parte do resultado.

Gráficos devem ser acompanhados de interpretação. O objetivo é demonstrar que o aluno entende a relação entre algoritmo, runtime OpenMP e arquitetura multicore.

# 38. Checklist de domínio de OpenMP

Ao final desta parte, o aluno deve conseguir explicar e usar:

- [ ] Fork–Join
- [ ] `parallel`
- [ ] `num_threads`
- [ ] `parallel for`
- [ ] `schedule(static)`
- [ ] `schedule(dynamic)`
- [ ] `schedule(guided)`
- [ ] `schedule(runtime)`
- [ ] `shared`
- [ ] `private`
- [ ] `firstprivate`
- [ ] `lastprivate`
- [ ] `default(none)`
- [ ] race conditions
- [ ] `critical`
- [ ] `atomic`
- [ ] `reduction`
- [ ] `barrier`
- [ ] `nowait`
- [ ] `ordered`
- [ ] `single`
- [ ] `master`
- [ ] `masked`
- [ ] `sections`
- [ ] `collapse`
- [ ] `simd`
- [ ] `simd reduction`
- [ ] `parallel for simd`
- [ ] relatório de vetorização
- [ ] `task`
- [ ] `taskwait`
- [ ] `taskgroup`
- [ ] granularidade de tasks
- [ ] task dependencies
- [ ] `taskloop`
- [ ] locks
- [ ] `OMP_PROC_BIND`
- [ ] `OMP_PLACES`
- [ ] false sharing
- [ ] localidade
- [ ] paralelismo aninhado
- [ ] speedup
- [ ] eficiência
- [ ] Lei de Amdahl
- [ ] strong scaling
- [ ] weak scaling
- [ ] protocolo experimental reproduzível

### Como usar o checklist

O checklist não significa apenas reconhecer a sintaxe. Para marcar um item, o aluno deve conseguir explicar o mecanismo, implementar um exemplo correto e identificar pelo menos uma situação em que o recurso é adequado.

Itens como `schedule`, SIMD, tasks e afinidade também devem ser associados ao tipo de gargalo que procuram resolver.

# 39. Referências e continuação

Referências recomendadas:

- OpenMP Architecture Review Board — **OpenMP API Specification 5.2**.
- OpenMP Architecture Review Board — **OpenMP Examples**.
- GCC — documentação de otimização e relatórios de vetorização.
- documentação da arquitetura da CPU utilizada no laboratório.

Este notebook privilegia mecanismos de OpenMP executáveis em CPU com GCC, inclusive em Google Colab.

A continuação natural da disciplina é reutilizar o mesmo padrão experimental na parte de CUDA:

**baseline → corretude → decomposição → execução paralela → memória → sincronização → otimização → speedup → análise de gargalo.**

### Como aprofundar

A especificação OpenMP define formalmente o comportamento dos construtos. O documento de exemplos é normalmente mais acessível para aprender padrões de uso.

Quando o comportamento observado no laboratório for diferente do esperado, consulte primeiro a especificação e a documentação do compilador/runtime antes de assumir que determinado pragma “não funciona”.